In [ ]:
from itertools import combinations,permutations,product,chain
from operator import itemgetter
from math import comb,factorial

"""
Partial Matching between [n] and [m]
Is a map from subset of [n] to subset of [m] with sign +1 or -1
"""

def getz(I,i,d):
    if i < 0  or i >= len(I):
        return d
    else:
        return I[i]
    

def PM(m,n,k):
    assert(k <= min(n,m))
    #generate all subset of [n] with size k
    for domain in combinations(range(1,m+1), k):
        for codomain in permutations(range(1,n+1), k):
                yield frozenset(zip(sorted(domain),codomain))


def OPM(m,n,k):
    """
    Generates all ordered partial matching.
    It is given by I subset [m], J subset [n] such that |I| = |J|. 
    """
    assert(k <= min(n,m))
    #generate all subset of [n] with size k
    for domain in combinations(range(1,m+1), k):
        for codomain in combinations(range(1,n+1), k):
                yield (tuple(sorted(domain)),tuple(sorted(codomain)))


"""
Given a pair (I,J) such that 
|I| = |J| = k 
I \subset [m] and J subset [n]

It defines a B_1 x B_2 invariant subspace V_{I,J} in Mat_mxn 

...*x*********
...o**********
......*x******
......o*******
.........**x**
.........*x***
.........o****
..............
..............

where 
(I[i], J[i]) indicate the corner marked by 'o'.  
"""

def Vgenerator(m,n,k):
    """
    The generator of the vector space rank k level
    """
    return (tuple(range(m-k+1,m+1)),tuple(range(1,k+1)))

def PMgen(m,n,k):
    """
    The generator of the partial matching rank k level
    """
    return frozenset(zip(range(m-k+1,m+1),(range(1,k+1))))


def basis(m,n,sigma):
    """
    basis of the space .  
    """
    I , J = sigma
    A = []
    for a in range(1,m+1):
        m = 0
        for k in range(len(I)):
            if I[k] <= a:
                m = J[k]
        for b in range(1,m+1):
            A.append((a,b))
    return frozenset(A) 

def OPMdim(m,n,sigma):
    B = basis(m,n,sigma) 
    return len(B)

def perpbasis(m,n,B):
    A = []
    for c in range(1,m+1):
        for d in range(1,n+1):
            if (c,d) not in B:
                A.append((d,c))
    return frozenset(A)
    


def Fourier(m,n,sigma):
    I,J = sigma
    if 1 in I:
        Q =  tuple(a+1 for a in J if a < n ) 
    else:
        Q = tuple(chain((1,), (a+1 for a in J if a < n )))
    if n in J:
        P =  tuple(a-1 for a in I if 1 < a) 
    else:
        P = tuple(chain((a-1 for a in I if 1<a ),(m,)))
    return (Q,P)


def AOPM(m,n):
    """
    All ordered partial matching
    """
    for k in range(0,min(m,n)+1):
        for sigma in OPM(m,n,k):
            yield sigma




def sizeallpm(m,n):
    """
    The size of all partial matching
    = sum_k |S_m/S_(m-k) x S_k| |S_n/S_(n-k)| 
    """
    return sum(comb(m,k)*comb(n,k)*factorial(k) for k in range(min(m,n)+1))






In [ ]:
"""
s_i=(i,i+1) in type A simple root of S_m  
"""

def si(m,i):
    return tuple(chain(range(1,i), (i+1,i),range(i+2,m+1)))

def simplerefA(m):
    for i in range(1,m):
        yield si(m,i)




def leftcosetA(n,k):
    """
    Generate the minimal length representative of S_n / S_(n-k) x S_k 
    """
    assert(k<=n)
    A = list(range(1,n+1))
    for r in combinations(A,n-k):
        r = sorted(r)
        yield tuple(chain(r,(a for a in A if a not in r)))

def leftcosetB(n,k):
    """
    Generate the minimal length representative of S_n / 1 x S_(n-k) 
    """
    assert(k<=n)
    A = list(range(1,n+1))
    for r in combinations(A,k):
        r = sorted(r)
        rc = tuple(a for a in A if a not in r)
        for rr in permutations(r):
            yield tuple(chain(rr,rc))

def Aact(perm,sigma):
    """
    Apply the perm action of W1 on sigma 
    """
    nsigma = frozenset((perm[s[0]-1],s[1]) for s in sigma)
    return nsigma

def Bact(perm,sigma):
    """
    Apply the perm action of W2 on sigma 
    """
    nsigma = frozenset((s[0],perm[s[1]-1]) for s in sigma)
    return nsigma

def lengthA(perm): 
    """
    A permutation is a ordered list of positive integers.
    length is the inversion number of the permutation.
    """
    return sum(1 for i in range(len(perm)) for j in range(i+1, len(perm)) if perm[i] > perm[j])

def cleverlength(m,n,k):
    D = dict()
    gen = PMgen(m,n,k)
    d0 = OPMdim(m,n, Vgenerator(m,n,k))
    #print(f"Vgenerator {Vgenerator(m,n,k)} dim : {d0}")
    for p1 in leftcosetA(m,k):
        nsigma1 = Aact(p1,gen)
        for p2 in leftcosetB(n,k):
            nsigma = Bact(p2,nsigma1)
            l = lengthA(p1) + lengthA(p2)+d0
            #print(f"sigma: {nsigma}")
            #print(f"{p1}:{lengthA(p1)}, {p2}:{lengthB(p2)}")
            D[nsigma] = l
    return D
 
def lengthdict(m,n):
    D = dict()
    for k in range(min(m,n)+1):
        D.update(cleverlength(m,n,k)) 
    return D

def toPM(m,n, sigma):
    I, J = sigma 
    assert(len(I)==len(J))
    L = []
    R = []
    r = 0
    for i in range(len(I)):
        L = list(range(J[i],r,-1))+L
        r = J[i]
        for x in range(I[i],getz(I,i+1,m+1)):
            if len(L)>0:
                y = L.pop(0)
                R.append((x,y))
            else:
                break
    return frozenset(R)

def toPMp(m,n, sigma):
    I, J = sigma 
    assert(len(I)==len(J))
    Q = []
    R = []
    for k in range(len(I),0,-1):
        k = k-1
        Q = list(range(I[k],getz(I,k+1,m+1)))+Q
        for y in range(J[k],getz(J,k-1,0),-1):
            if len(Q)>0:
                x = Q.pop(0)
                R.append((x,y))
            else:
                break
    return frozenset(R)


def testmap(m,n):
    print(f"Test ({m},{n})")
    A = set(AOPM(m,n))
    B = set(AOPM(n,m))
    #print(f"A: {A}")
    #print(f"B: {B}")
    print(f"All partial matching {sizeallpm(m,n)}")
    FA = set(Fourier(m,n,sigma) for sigma in A)
    #relevent partial matching
    for mu in AOPM(m,n):
        sigma1 = toPM(m,n,mu)
        sigma2 = toPMp(m,n,mu)
        if sigma1!=sigma2:
            print(f"{mu} => {sigma1} != {sigma2}")
            break
    RPM = set(toPM(m,n,sigma) for sigma in AOPM(m,n))
    #print(RPM)
    print(f"PM {len(RPM)} = {len(A)}")
    for sigma in AOPM(m,n):

        fa = Fourier(m,n,sigma)
        print(f"Orig: {sigma} -> {toPM(m,n,sigma)}")
        print(f"Four {fa} -> {toPM(n,m,fa)}")


        fb = Fourier(n,m,fa)
        if fb != sigma:
            print(f"{sigma} -> {fa}->{fb}")
        Bsigma = basis(m,n,sigma)
        Psigma = perpbasis(m,n,Bsigma)
        Fbasis = basis(n,m,fa)
        if Fbasis != Psigma:
            print(f"{sigma} -> {fa}->{fb}")
            print(f"Basis: {Bsigma}")
    print(f"{len(A)},{len(B)}, {len(FA)}")
    print(f"B-FA:{B-FA}")
    print(f"FA-B:{FA-B}")

        


In [ ]:
def testdim(m,n):
    D = lengthdict(m,n)
    for k in range(min(m,n)+1):
        for sigma in OPM(m,n,k):
            pmsigma = toPM(m,n,sigma) 
            l = D[pmsigma]
            d = OPMdim(m,n,sigma)
            if l!=d:
                print(f"m:{m}, n:{n}, k:{k}, sigma:{sigma}, l:{l}, d:{d}")

"""
The following Fourier map is not correct.
"""
def genFourier(m,n):
    D = lengthdict(m,n)
    Dp = lengthdict(n,m)
    #print(f"D:{D}")
    #print(f"Dp:{Dp}")
    APM = set(D.keys())
    F = dict()
    for k in range(min(m,n)+1):
        for mu in OPM(m,n,k):
            sigma = toPM(m,n,mu)
            mup = Fourier(m,n,mu)
            sigmap = toPM(n,m,mup)
            F[sigma] = sigmap
            APM.remove(sigma)
    print(f"OPM : {len(F)} : Rest of PM :{len(APM)}")
    flag = True 
    while flag:
        flag = False
        for s1 in simplerefA(m):
            RM = []
            for sigma in APM: 
                sigma1 = Aact(s1,sigma) 
                assert(sigma == Aact(s1,sigma1))
                if sigma1 in F and D[sigma1] < D[sigma]:
                    sigmap1 = F[sigma1]
                    sigmap = Bact(s1,sigmap1)
                    assert(sigmap1 == Bact(s1,sigmap))
                    if Dp[sigmap1]< Dp[sigmap]:
                        F[sigma] = sigmap
                        RM.append(sigma) 
                        flag = True
                    else:
                        print(f"sigma {sigma} sigma1 {sigma1} sigmap {sigmap} sigmap1 {sigmap1}")
            for sigma in RM:
                APM.remove(sigma)
                    
        for s2 in simplerefA(n):
            RM = []
            for sigma in APM: 
                sigma1 = Bact(s2,sigma) 
                assert(sigma == Bact(s2,sigma1))
                if sigma1 in F and D[sigma1]< D[sigma]:
                    sigmap1 = F[sigma1]
                    sigmap = Aact(s2,sigmap1)
                    assert(sigmap1 == Aact(s2,sigmap))
                    if Dp[sigmap1]< Dp[sigmap]:
                        F[sigma] = sigmap
                        RM.append(sigma) 
                        flag = True
                    else:
                        print(f"sigma {sigma} sigma1 {sigma1} sigmap {sigmap} sigmap1 {sigmap1}")

            for sigma in RM:
                APM.remove(sigma)             
    print(f"All PM : {len(D.keys())}, F: {len(F.keys())}")
    assert(all(F[sigma] in Dp for sigma in F))
    IF = set(F[sigma] for sigma in F)
    print(f"Image of Fourier: {len(IF)}")

    


In [ ]:
m, n, k = 8,3,2
testmap(m,n)

In [ ]:
m, n, k = 8,3,2

#testdim(m,n)
#testmap(m,n)
genFourier(m,n)